In [0]:
from pyspark.sql.functions import col, to_timestamp, regexp_replace, when

catalog = "jarvis_workspace"

transactions_df = spark.table(f"{catalog}.bronze.transactions_data")
cards_df = spark.table(f"{catalog}.bronze.cards_data")
users_df = spark.table(f"{catalog}.bronze.users_data")
mcc_df = spark.table(f"{catalog}.bronze.mcc_codes")
fraud_df = spark.table(f"{catalog}.bronze.train_fraud_labels")

In [0]:
transactions_df.printSchema()
cards_df.printSchema()
users_df.printSchema()
fraud_df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- date: timestamp (nullable = true)
 |-- client_id: short (nullable = true)
 |-- card_id: short (nullable = true)
 |-- amount: decimal(19,4) (nullable = true)
 |-- use_chip: string (nullable = true)
 |-- merchant_id: integer (nullable = true)
 |-- merchant_city: string (nullable = true)
 |-- merchant_state: string (nullable = true)
 |-- zip: double (nullable = true)
 |-- mcc: short (nullable = true)
 |-- errors: string (nullable = true)

root
 |-- id: short (nullable = true)
 |-- client_id: short (nullable = true)
 |-- card_brand: string (nullable = true)
 |-- card_type: string (nullable = true)
 |-- card_number: long (nullable = true)
 |-- expires: string (nullable = true)
 |-- cvv: short (nullable = true)
 |-- has_chip: boolean (nullable = true)
 |-- num_cards_issued: short (nullable = true)
 |-- credit_limit: decimal(19,4) (nullable = true)
 |-- acct_open_date: string (nullable = true)
 |-- year_pin_last_changed: short (nullable = true)
 |-

In [0]:
display(transactions_df.limit(5))
display(fraud_df.limit(5))
display(mcc_df.limit(5))

id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors
9957427,2011-08-20T15:37:00.000Z,1797,1127,53.0700,Swipe Transaction,60569,Pleasanton,CA,94588.0,5300,null
9957428,2011-08-20T15:38:00.000Z,331,3818,60.9500,Swipe Transaction,60569,Aynor,SC,29511.0,5300,Insufficient Balance
9957429,2011-08-20T15:38:00.000Z,356,5605,29.1600,Swipe Transaction,65965,Stuart,FL,34997.0,5300,null
9957430,2011-08-20T15:38:00.000Z,489,5069,8.6400,Swipe Transaction,74734,York,NE,68467.0,5411,null
9957431,2011-08-20T15:38:00.000Z,561,4575,156.1200,Swipe Transaction,54850,Davenport,IA,52804.0,4814,null


transaction_id,is_fraud
10649266,No
23410063,No
9316588,No
12478022,No
9558530,No


mcc,description
1711,"Heating, Plumbing, Air Conditioning Contractors"
3000,Steelworks
3001,Steel Products Manufacturing
3005,Miscellaneous Metal Fabrication
3006,Miscellaneous Fabricated Metal Products


In [0]:
from pyspark.sql.functions import col, to_timestamp, when

catalog = "jarvis_workspace"

transactions_df = spark.table(f"{catalog}.bronze.transactions_data")
cards_df = spark.table(f"{catalog}.bronze.cards_data")
users_df = spark.table(f"{catalog}.bronze.users_data")
mcc_df = spark.table(f"{catalog}.bronze.mcc_codes")
fraud_df = spark.table(f"{catalog}.bronze.train_fraud_labels")

silver_transactions = (
    transactions_df
    .withColumn("transaction_ts", to_timestamp(col("date")))
    .withColumn("amount", col("amount").cast("double"))
    .withColumn("mcc", col("mcc").cast("string"))
    .join(
        fraud_df.withColumn("transaction_id", col("transaction_id").cast("long")),
        transactions_df["id"] == col("transaction_id"),
        "left"
    )
    .join(
        mcc_df.withColumn("mcc", col("mcc").cast("string")),
        "mcc",
        "left"
    )
    .withColumn("is_fraud", when(col("is_fraud") == "Yes", True).otherwise(False))
    .drop("transaction_id")
    .dropDuplicates()
)

silver_cards = cards_df.dropDuplicates()
silver_users = users_df.dropDuplicates()

silver_transactions.write.mode("overwrite").saveAsTable(f"{catalog}.silver.transactions")
silver_cards.write.mode("overwrite").saveAsTable(f"{catalog}.silver.cards")
silver_users.write.mode("overwrite").saveAsTable(f"{catalog}.silver.users")

In [0]:
display(spark.table("jarvis_workspace.silver.transactions").limit(10))
spark.sql("SHOW TABLES IN jarvis_workspace.silver").show(truncate=False)

mcc,id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,errors,transaction_ts,is_fraud,description
5541,10297307,2011-11-07T00:34:00.000Z,157,1052,8.92,Swipe Transaction,96049,Perris,CA,92571.0,null,2011-11-07T00:34:00.000Z,false,Service Stations
7538,10321300,2011-11-12T11:59:00.000Z,213,2136,93.49,Swipe Transaction,71667,Garland,TX,75044.0,null,2011-11-12T11:59:00.000Z,false,Automotive Service Shops
5912,10472958,2011-12-17T10:13:00.000Z,1849,5904,59.05,Swipe Transaction,20561,Tacoma,WA,98445.0,null,2011-12-17T10:13:00.000Z,false,Drug Stores and Pharmacies
5499,10515852,2011-12-27T06:28:00.000Z,1207,5805,5.02,Swipe Transaction,59935,Cleveland,OH,44143.0,null,2011-12-27T06:28:00.000Z,false,Miscellaneous Food Stores
5499,7501471,2010-01-07T14:35:00.000Z,782,2216,53.61,Swipe Transaction,43293,Defiance,OH,43512.0,null,2010-01-07T14:35:00.000Z,false,Miscellaneous Food Stores
6300,7523397,2010-01-13T09:25:00.000Z,615,1128,92.77,Swipe Transaction,1567,Alvin,TX,77511.0,null,2010-01-13T09:25:00.000Z,false,"Insurance Sales, Underwriting"
5912,7624318,2010-02-08T04:39:00.000Z,543,4708,8.99,Swipe Transaction,20561,Coos Bay,OR,97420.0,null,2010-02-08T04:39:00.000Z,false,Drug Stores and Pharmacies
5310,7986880,2010-05-09T15:32:00.000Z,638,3394,100.78,Swipe Transaction,39324,Blair,NE,68008.0,null,2010-05-09T15:32:00.000Z,false,Discount Stores
5411,8143866,2010-06-17T08:02:00.000Z,565,5459,5.17,Swipe Transaction,3531,Carrollton,TX,75006.0,null,2010-06-17T08:02:00.000Z,false,"Grocery Stores, Supermarkets"
5499,8183865,2010-06-27T06:06:00.000Z,760,5876,2.06,Swipe Transaction,59935,Columbus,OH,43228.0,null,2010-06-27T06:06:00.000Z,false,Miscellaneous Food Stores


+--------+------------+-----------+
|database|tableName   |isTemporary|
+--------+------------+-----------+
|silver  |cards       |false      |
|silver  |transactions|false      |
|silver  |users       |false      |
+--------+------------+-----------+

